# Cleaned Parquet Explorer

Use this notebook to preview the cleaned parquet files and get quick stats (shape, columns, missingness).

In [11]:
from pathlib import Path
import pandas as pd

# Robust base path whether notebook is run from repo root or datasets/
cwd = Path.cwd()
if cwd.name == 'datasets':
    BASE_DIR = cwd / 'parquet_by_domain'
else:
    BASE_DIR = cwd / 'datasets' / 'parquet_by_domain'

parquet_files = sorted(BASE_DIR.glob('*.parquet'))
parquet_files

[PosixPath('/Users/chasecapanna/MIDS/DATASCI210/PROJECT/co_claims/datasets/parquet_by_domain/cnbc_com_cleaned.parquet'),
 PosixPath('/Users/chasecapanna/MIDS/DATASCI210/PROJECT/co_claims/datasets/parquet_by_domain/cnn_com_cleaned.parquet'),
 PosixPath('/Users/chasecapanna/MIDS/DATASCI210/PROJECT/co_claims/datasets/parquet_by_domain/finance_yahoo_com_cleaned.parquet'),
 PosixPath('/Users/chasecapanna/MIDS/DATASCI210/PROJECT/co_claims/datasets/parquet_by_domain/nasdaq_com_cleaned.parquet')]

In [12]:
# Pick a file to inspect
FILE = 'cnbc_com_cleaned.parquet'
path = BASE_DIR / FILE
path

PosixPath('/Users/chasecapanna/MIDS/DATASCI210/PROJECT/co_claims/datasets/parquet_by_domain/cnbc_com_cleaned.parquet')

In [13]:
df = pd.read_parquet(path)
df.head(5)

,cik,ticker,accession_number,form_type,filing_date,section,fact_type,direction,evidence_text,source_url,...,language,sourcecountry,text,source_file,news_site,title_date,text_date,headline_direction,text_direction,company_name
0,1026785.0,HIHO,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,HIGHWAY HOLDINGS LTD
1,1823652.0,EVEX,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,"Eve Holding, Inc."
2,1823652.0,EVEX-WT,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,"Eve Holding, Inc."
3,1564708.0,NWSA,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,NEWS CORP
4,1564708.0,NWS,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,NEWS CORP


In [14]:
df.shape

(236722, 27)

In [15]:
df.columns

Index(['cik', 'ticker', 'accession_number', 'form_type', 'filing_date',
       'section', 'fact_type', 'direction', 'evidence_text', 'source_url',
       'sent_index', 'url', 'url_mobile', 'title', 'seendate', 'socialimage',
       'domain', 'language', 'sourcecountry', 'text', 'source_file',
       'news_site', 'title_date', 'text_date', 'headline_direction',
       'text_direction', 'company_name'],
      dtype='str')

In [16]:
# Quick missingness scan
missing = df.isna().mean().sort_values(ascending=False)
missing.head(15)

sent_index            1.000000
accession_number      1.000000
title_date            0.999632
text_date             0.893732
filing_date           0.893656
url_mobile            0.061933
socialimage           0.031822
cik                   0.022765
ticker                0.022765
company_name          0.022765
source_url            0.000000
text                  0.000000
text_direction        0.000000
headline_direction    0.000000
form_type             0.000000
dtype: float64

In [17]:
# Optional: basic stats for numeric columns
df.select_dtypes(include='number').describe().T

,count,mean,std,min,25%,50%,75%,max
cik,231333.0,1.222708e+06,568443.236811,1750.0,895421.0,1163653.0,1586554.0,2107523.0


## Combined Sentiment Distribution

This aggregates all sources and plots the count of `direction` labels.

In [ ]:
# Aggregate sentiment counts per article without combining full parquets
sentiment_order = ['negative', 'neutral', 'positive']
article_sentiment_sums = []

for f in parquet_files:
    try:
        df = pd.read_parquet(f, columns=['url', 'title', 'direction'])
    except Exception as e:
        print(f'Skipping {f.name}: {e}')
        continue

    article_key = 'url' if 'url' in df.columns else 'title'
    if article_key not in df.columns or 'direction' not in df.columns:
        print(f'Skipping {f.name}: missing required columns')
        continue

    tmp = df[[article_key, 'direction']].copy()
    tmp['direction'] = tmp['direction'].astype(str).str.lower()
    tmp = tmp.dropna(subset=[article_key, 'direction'])

    counts = (
        tmp.groupby([article_key, 'direction'])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=sentiment_order, fill_value=0)
    )
    article_sentiment_sums.append(counts)

# Combine only the per-article aggregates (small), then total for plotting
article_sentiment_sums_df = (
    pd.concat(article_sentiment_sums, axis=0)
      .groupby(level=0)
      .sum()
) if article_sentiment_sums else pd.DataFrame(columns=sentiment_order)

sentiment_totals = article_sentiment_sums_df.sum().reindex(sentiment_order, fill_value=0)
sentiment_totals


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Dark theme styling
plt.style.use('dark_background')

fig, ax = plt.subplots(facecolor='#0b0b0b')
ax.set_facecolor('#0b0b0b')

ax = sentiment_totals.plot(
    kind='bar',
    color=['#b23b2e', '#4b4f5c', '#2b7a6f'],
    ax=ax,
)
ax.set_title('Articles by Sentiment (All Sources)', color='#f0f0f0')
ax.set_xlabel('Sentiment', color='#f0f0f0')
ax.set_ylabel('Number of Articles (Summed by Article)', color='#f0f0f0')

# Y-axis as 100k, 200k, etc.
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1000:.0f}k' if x >= 1000 else f'{int(x)}'))

# Bar labels with comma separators
ax.bar_label(ax.containers[0], labels=[f'{int(v):,}' for v in sentiment_totals.values])

# Rotate x-axis labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Ticks styling
ax.tick_params(colors='#f0f0f0')

plt.tight_layout()
plt.show()
